In [1]:
from __future__ import annotations
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
from loguru import logger

ROOT = Path.cwd().resolve()

while ROOT != ROOT.parent and not (ROOT / "app").is_dir():
    ROOT = ROOT.parent

if not (ROOT / "app").is_dir():
    raise RuntimeError(
        "Папка app не найдена. "
        "Откройте Notebook внутри репозитория CloudMarks."
    )

sys.path.insert(0, str(ROOT))

In [2]:
from app.cross_points.CrossPointExacter import CrossPointExacter


exacter = CrossPointExacter(
    file_path="../data/t1/1_A_10_2_l.txt",
    show_scans=False,
    normal_k=8,
    min_points_per_plane=100,
    cluster_eps=0.08,
)

point = exacter.calculate_intersect_point()
point.name = "A_10_2_l"

df_point = point.to_dataframe()
display(df_point)

2026-08-29 12:32:17.845 | INFO     | app.cross_points.CrossPointExacter:_initialize:262 - Подготовка виртуальной точки: scan='1_A_10_2_l', N=5931, reference=None
2026-08-29 12:32:17.845 | INFO     | app.cross_points.CrossPointExacter:_segment_local_planes:299 - Вычисление нормалей: k=8
2026-08-29 12:32:17.921 | INFO     | app.cross_points.CrossPointExacter:_segment_local_planes:305 - Классификация направлений нормалей: n_classes=3
2026-08-29 12:32:17.955 | INFO     | app.cross_points.CrossPointExacter:_segment_local_planes:319 - Размеры классов нормалей: {0: 1111, 1: 4703, 2: 117}
2026-08-29 12:32:17.966 | DEBUG    | app.cross_points.CrossPointExacter:_select_component:442 - Плоскость 1: DBSCAN-компонента 0, точек=1111
2026-08-29 12:32:17.968 | INFO     | app.cross_points.CrossPointExacter:_segment_local_planes:356 - Плоскость 1/3: класс нормалей 0, выбрано 1111 точек
2026-08-29 12:32:18.004 | DEBUG    | app.cross_points.CrossPointExacter:_select_component:442 - Плоскость 2: DBSCAN-ком

,name,x,y,z,status,reliable_accuracy,mse,plane_1_mse,plane_2_mse,plane_3_mse,sigma_x,sigma_y,sigma_z,ellipsoid_confidence,ellipsoid_a,ellipsoid_b,ellipsoid_c
0,A_10_2_l,99.313348,107.766052,8.153626,GOOD,True,0.000189,0.000529,0.000794,0.000557,0.000179,0.000015,0.000057,0.95,0.000501,0.00016,0.000042


In [3]:
print(
    f"{point.name}: "
    f"X={point.x:.6f}, "
    f"Y={point.y:.6f}, "
    f"Z={point.z:.6f}, "
    f"status={point.status}"
)

print(point)

A_10_2_l: X=99.313348, Y=107.766052, Z=8.153626, status=GOOD
CrossPoint (name=A_10_2_l, status=GOOD, x=99.313348, y=107.766052, z=8.153626, plane_mses=[0.000529, 0.000794, 0.000557], mse=0.000189, sigma_xyz=(0.000179, 0.000015, 0.000057))


In [4]:
print(f"Имя точки: {point.name}")
print()

print("Координаты, м")
print(f"  X: {point.x:.6f}")
print(f"  Y: {point.y:.6f}")
print(f"  Z: {point.z:.6f}")
print()

print("Статус")
print(f"  Геометрия: {point.status}")
print(
    "  Точность: "
    f"{'надёжная' if point.reliable_accuracy else 'ненадёжная'}"
)
print()

if point.mse is not None:
    print("Средняя квадратическая ошибка")
    print(f"  RMSE: {point.mse:.6f} м")
    print()

if point.planes_mse is not None:
    print("RMSE аппроксимации плоскостей")

    for index, plane_mse in enumerate(
        point.planes_mse,
        start=1,
    ):
        print(
            f"  Плоскость {index}: "
            f"{plane_mse:.6f} м"
        )

    print()

if point.reliable_accuracy and point.sigma_xyz is not None:
    sigma_x, sigma_y, sigma_z = point.sigma_xyz

    print("Средние квадратические погрешности координат")
    print(f"  σX: {sigma_x:.6f} м")
    print(f"  σY: {sigma_y:.6f} м")
    print(f"  σZ: {sigma_z:.6f} м")
    print()

if point.reliable_accuracy and point.cov_xyz is not None:
    print("Ковариационная матрица координат, м²")
    print(point.cov_xyz)
    print()

if point.reliable_accuracy and point.ellipsoid is not None:
    semi_axes = point.ellipsoid["semi_axes"]
    confidence = point.ellipsoid["confidence"]

    print(
        "Эллипсоид погрешностей "
        f"(доверительная вероятность {confidence:.0%})"
    )
    print(f"  Большая полуось: {semi_axes[0]:.6f} м")
    print(f"  Средняя полуось: {semi_axes[1]:.6f} м")
    print(f"  Малая полуось:   {semi_axes[2]:.6f} м")

Имя точки: A_10_2_l

Координаты, м
  X: 99.313348
  Y: 107.766052
  Z: 8.153626

Статус
  Геометрия: GOOD
  Точность: надёжная

Средняя квадратическая ошибка
  RMSE: 0.000189 м

RMSE аппроксимации плоскостей
  Плоскость 1: 0.000529 м
  Плоскость 2: 0.000794 м
  Плоскость 3: 0.000557 м

Средние квадратические погрешности координат
  σX: 0.000179 м
  σY: 0.000015 м
  σZ: 0.000057 м

Ковариационная матрица координат, м²
[[ 3.21442012e-08 -3.36973585e-11 -9.24727633e-11]
 [-3.36973585e-11  2.30164278e-10  1.46813357e-11]
 [-9.24727633e-11  1.46813357e-11  3.28643337e-09]]

Эллипсоид погрешностей (доверительная вероятность 95%)
  Большая полуось: 0.000501 м
  Средняя полуось: 0.000160 м
  Малая полуось:   0.000042 м


In [5]:
output_path = ROOT / "output" / f"{point.name}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_point.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
    float_format="%.6f",
)

df_point
output_path

PosixPath('/Users/mihailvystrcil/Documents/My Programms/CloudMarks/output/A_10_2_l.csv')

In [6]:
output_path = ROOT / "output" / f"{point.name}.xlsx"
output_path.parent.mkdir(parents=True, exist_ok=True)

point.to_dataframe().to_excel(
    output_path,
    index=False,
    sheet_name="VirtualPoint",
)

print(f"Excel-файл сохранён: {output_path}")

Excel-файл сохранён: /Users/mihailvystrcil/Documents/My Programms/CloudMarks/output/A_10_2_l.xlsx


In [7]:
from loguru import logger
from app.batch.SingleScanPointExtractor import (
    SingleScanPointExtractor,
)

logger.remove()

extractor = SingleScanPointExtractor.from_files(
    scan_path="../data/t2/scan_2335.las",
    reference_points_path="../data/t2/vse_tochki.txt",
    default_radius=0.25,
    min_neighborhood_points=800,
    min_points_per_plane=150,
    normal_k=8,
    cluster_eps=0.08,
)

results = extractor.run(
    show_progress=True
).to_dataframe()

results


[1/3] Загрузка скана... 

Загрузка scan_2335.las: 100%|██████████| 4337974/4337974 [00:10<00:00, 399295.89точка/s]

готово: 4,337,974 точек
[2/3] Построение пространственного индекса... 

готово
Опорных точек: 196


[3/3] Извлечение точек: 100%|██████████| 196/196 [00:09<00:00, 19.60точка/s]

Готово: 35/196 надёжных точек


,name,reference_x,reference_y,reference_z,radius,neighborhood_points,reference_distance,status,message,x,y,z,geometry_status,reliable_accuracy,sigma_x,sigma_y,sigma_z
0,A_10_2_l,99.3127,107.7533,8.1465,0.25,4682,NaN,UNRELIABLE,Для плоскости 3 выделено недостаточно точек: 9...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
1,A_10_2_r,101.0595,107.7900,8.1438,0.25,4798,NaN,UNRELIABLE,Точка ненадёжна: geometry_status=PARALLEL.,100.516326,107.765778,8.154636,PARALLEL,False,NaN,NaN,NaN
2,A_10_3_l,99.3061,107.7612,11.6365,0.25,3261,NaN,UNRELIABLE,Точка ненадёжна: geometry_status=PARALLEL.,99.677008,107.764067,11.646920,PARALLEL,False,NaN,NaN,NaN
3,A_10_3_r,101.0720,107.7561,11.6403,0.25,3303,0.175203,SUCCESS,OK,100.897215,107.765188,11.648275,GOOD,True,0.005155,0.000046,0.000063
4,A_10_4_l,99.2973,107.7508,15.3065,0.25,2330,NaN,UNRELIABLE,Для плоскости 3 выделено недостаточно точек: 4...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,A_9_6_r,98.0572,107.7900,22.4644,0.25,1247,NaN,UNRELIABLE,Для плоскости 3 выделено недостаточно точек: 5...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
192,A_9_7_l,96.2911,107.7365,25.9726,0.25,1043,NaN,UNRELIABLE,Для плоскости 1 выделено недостаточно точек: 9...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
193,A_9_7_r,98.0578,107.7522,25.9854,0.25,1007,NaN,UNRELIABLE,Для плоскости 3 выделено недостаточно точек: 3...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN
194,A_9_8_l,96.2968,107.7472,29.5982,0.25,914,NaN,UNRELIABLE,Для плоскости 1 выделено недостаточно точек: 6...,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN


In [8]:
points = results.query("status == 'SUCCESS'")[
    [
        "name",
        "x",
        "y",
        "z",
        "sigma_x",
        "sigma_y",
        "sigma_z",
    ]
]

points

,name,x,y,z,sigma_x,sigma_y,sigma_z
3,A_10_3_r,100.897215,107.765188,11.648275,0.005155,0.000046,0.000063
15,A_11_2_r,104.086872,107.765617,8.155802,0.000107,0.000020,0.000088
17,A_11_3_r,104.087458,107.760094,11.644738,0.000091,0.000040,0.000073
19,A_11_4_r,104.087433,107.758718,15.326739,0.000163,0.000030,0.000116
21,A_11_5_r,104.087030,107.749718,18.848946,0.000153,0.000036,0.000110
29,A_12_2_r,107.086962,107.761244,8.143266,0.000100,0.000046,0.000145
31,A_12_3_r,107.086054,107.755811,11.650559,0.000111,0.000028,0.000138
33,A_12_4_r,107.097349,107.760624,15.330984,0.000122,0.000040,0.000107
43,A_13_2_r,110.097429,107.767612,8.154826,0.000124,0.000034,0.000102
45,A_13_3_r,110.098202,107.757693,11.642005,0.000124,0.000039,0.000112


In [9]:
output = ROOT / "output" / "t2_virtual_points.xlsx"
output.parent.mkdir(exist_ok=True)

points.to_excel(
    output,
    index=False,
    sheet_name="VirtualPoints",
)

print(f"Найдено точек: {len(points)}")
print(f"Файл: {output}")

Найдено точек: 35
Файл: /Users/mihailvystrcil/Documents/My Programms/CloudMarks/output/t2_virtual_points.xlsx


In [10]:
from loguru import logger
from app.batch.PointPairComparisonRunner import (
    PointPairComparisonRunner,
)

logger.remove()
runner = PointPairComparisonRunner.from_files(
    epoch1_path="../data/t2/scan_2335_split_a.las",
    epoch2_path="../data/t2/scan_2335_split_b.las",
    default_radius=0.5,
    min_neighborhood_points=800,
    min_points_per_plane=200,
    normal_k=8,
    cluster_eps=0.08,
)

results = runner.run_from_reference_file(
    reference_points_path="../data/t2/vse_tochki.txt",
    show_progress=True,
)

deformations = results.query(
    "processing_status == 'SUCCESS'"
)

deformations[
    [
        "name",
        "epoch1_x",
        "epoch1_y",
        "epoch1_z",
        "epoch2_x",
        "epoch2_y",
        "epoch2_z",
        "dx",
        "dy",
        "dz",
        "displacement_mm",
    ]
]

Обработка точек: 100%|██████████| 392/392 [00:32<00:00, 11.92эпоха/s, OK=22]

Анализ деформаций... готово: 22/196 пар


,name,epoch1_x,epoch1_y,epoch1_z,epoch2_x,epoch2_y,epoch2_z,dx,dy,dz,displacement_mm
15,A_11_2_r,104.086941,107.765804,8.155451,104.087232,107.765756,8.154809,0.000291,-0.000048,-0.000642,0.705993
17,A_11_3_r,104.087522,107.761363,11.645304,104.087849,107.761343,11.645753,0.000326,-0.000021,0.000449,0.555403
19,A_11_4_r,104.086554,107.758426,15.326991,104.087550,107.758429,15.327224,0.000996,0.000003,0.000233,1.022641
29,A_12_2_r,107.086856,107.764275,8.144277,107.086757,107.764300,8.144054,-0.000100,0.000025,-0.000222,0.244881
35,A_12_5_r,107.104637,107.758165,18.859744,107.104598,107.758931,18.859190,-0.000039,0.000765,-0.000553,0.945240
43,A_13_2_r,110.095698,107.767389,8.154570,110.095993,107.767510,8.154915,0.000294,0.000121,0.000346,0.469925
47,A_13_4_r,110.094647,107.758220,15.325413,110.094444,107.758247,15.325274,-0.000203,0.000027,-0.000140,0.247795
49,A_13_5_r,110.100289,107.759545,18.857836,110.099659,107.759429,18.857636,-0.000630,-0.000116,-0.000200,0.670625
59,A_14_3_r,113.107032,107.758719,11.639462,113.107167,107.758788,11.640212,0.000135,0.000069,0.000750,0.764873
61,A_14_4_r,113.111217,107.756944,15.323205,113.112512,107.756972,15.322930,0.001295,0.000028,-0.000275,1.323787


In [11]:
deformations

,name,reference_x,reference_y,reference_z,radius,epoch1_neighborhood_points,epoch2_neighborhood_points,epoch1_reference_distance,epoch2_reference_distance,pair_distance,...,sigma_dz,sigma_displacement,sigma_displacement_mm,t_value,p_value_t,significant_t,chi2_value,p_value_chi2,significant_chi2,analysis_reliable
15,A_11_2_r,104.0839,107.7565,8.1510,0.5,7019,6981,0.010753,0.010549,0.000706,...,0.000117,0.000118,0.118393,5.963117,2.474709e-09,True,40.481242,8.424288e-09,True,True
17,A_11_3_r,104.0912,107.7559,11.6290,0.5,4984,5030,0.017584,0.017931,0.000555,...,0.000117,0.000111,0.111380,4.986543,6.146928e-07,True,25.757542,1.071979e-05,True,True
19,A_11_4_r,104.0877,107.7516,15.3142,0.5,3412,3463,0.014543,0.014707,0.001023,...,0.000121,0.000190,0.189858,5.386344,7.190515e-08,True,29.927446,1.429415e-06,True,True
29,A_12_2_r,107.0840,107.7573,8.1347,0.5,4887,4877,0.012187,0.012004,0.000245,...,0.000114,0.000114,0.114385,2.140853,3.228593e-02,True,5.064232,1.671537e-01,False,True
35,A_12_5_r,107.1023,107.7518,18.8447,0.5,2213,2212,0.016501,0.016313,0.000945,...,0.000207,0.000140,0.140275,6.738465,1.600686e-11,True,96.724894,7.862990e-21,True,True
43,A_13_2_r,110.0912,107.7668,8.1473,0.5,3142,3186,0.008569,0.009026,0.000470,...,0.000131,0.000149,0.149384,3.145745,1.656642e-03,True,22.755789,4.540363e-05,True,True
47,A_13_4_r,110.0968,107.7900,15.3104,0.5,2081,2083,0.035214,0.035143,0.000248,...,0.000209,0.000191,0.190989,1.297429,1.944836e-01,False,1.942935,5.843349e-01,False,True
49,A_13_5_r,110.0993,107.7900,18.8398,0.5,1703,1701,0.035409,0.035395,0.000671,...,0.000234,0.000273,0.272601,2.460097,1.388996e-02,True,9.871257,1.969281e-02,True,True
59,A_14_3_r,113.1109,107.7541,11.6370,0.5,1858,1844,0.006509,0.006799,0.000765,...,0.000255,0.000253,0.253224,3.020536,2.523276e-03,True,10.205284,1.689938e-02,True,True
61,A_14_4_r,113.1084,107.7900,15.3194,0.5,1591,1589,0.033393,0.033470,0.001324,...,0.000240,0.000226,0.226299,5.849715,4.924170e-09,True,34.359909,1.663166e-07,True,True


In [12]:
deformations = (
    results.query("processing_status == 'SUCCESS'")
    .sort_values(
        by="displacement_mm",
        ascending=False,
        kind="stable",
    )
    .reset_index(drop=True)
)

deformations[
    [
        "name",
        "epoch1_x",
        "epoch1_y",
        "epoch1_z",
        "epoch2_x",
        "epoch2_y",
        "epoch2_z",
        "dx",
        "dy",
        "dz",
        "displacement_mm",
    ]
].round(4)

,name,epoch1_x,epoch1_y,epoch1_z,epoch2_x,epoch2_y,epoch2_z,dx,dy,dz,displacement_mm
0,A_14_4_r,113.1112,107.7569,15.3232,113.1125,107.7570,15.3229,0.0013,0.0000,-0.0003,1.3238
1,A_8_4_l,93.2885,107.7574,15.3149,93.2897,107.7575,15.3153,0.0012,0.0001,0.0003,1.2243
2,A_11_4_r,104.0866,107.7584,15.3270,104.0876,107.7584,15.3272,0.0010,0.0000,0.0002,1.0226
3,A_6_2_l,87.2774,107.7653,8.1464,87.2776,107.7651,8.1474,0.0002,-0.0001,0.0010,1.0164
4,A_12_5_r,107.1046,107.7582,18.8597,107.1046,107.7589,18.8592,-0.0000,0.0008,-0.0006,0.9452
5,A_15_4_r,116.1233,107.7615,15.3255,116.1233,107.7614,15.3264,-0.0000,-0.0001,0.0009,0.9284
6,A_14_3_r,113.1070,107.7587,11.6395,113.1072,107.7588,11.6402,0.0001,0.0001,0.0007,0.7649
7,A_8_5_l,93.2796,107.7565,18.8575,93.2799,107.7568,18.8569,0.0003,0.0003,-0.0006,0.7359
8,A_5_4_l,84.2748,107.7659,15.3270,84.2754,107.7659,15.3265,0.0006,-0.0000,-0.0004,0.7353
9,A_11_2_r,104.0869,107.7658,8.1555,104.0872,107.7658,8.1548,0.0003,-0.0000,-0.0006,0.7060
